In [ ]:
pip install vllm fastapi uvicorn pydantic requests pyngrok

In [ ]:
pip install "numpy<2.0.0" "scipy>=1.13.0"

In [ ]:
import os
from huggingface_hub import login


os.environ["HF_TOKEN"] = "YOUR_API_KEY"

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    hf_token = input("Enter your Hugging Face token: ").strip()

login(token=hf_token)

In [ ]:
import os
import uuid
import asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from contextlib import asynccontextmanager

from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
from vllm.sampling_params import SamplingParams

try:
    from vllm.sampling_params import StructuredOutputsParams
    HAS_STRUCTURED_PARAMS = True
except ImportError:
    HAS_STRUCTURED_PARAMS = False

engine = None
jobs = {}

@asynccontextmanager
async def lifespan(fastapi_app: FastAPI):
    global engine
    print("\n🚀 Inicializando vLLM en L4...")
    
    engine_args = AsyncEngineArgs(
        model="QuantTrio/Qwen3.5-9B-AWQ",    
        quantization="awq_marlin",           
        tensor_parallel_size=1,
        max_model_len=24576,
        gpu_memory_utilization=0.85,
        dtype="half",                        
        enforce_eager=False,
        skip_mm_profiling=True               
    )
    
    engine = AsyncLLMEngine.from_engine_args(engine_args)
    print("✅ Motor vLLM cargado en VRAM exitosamente. Listo para recibir peticiones.")
    yield

app = FastAPI(lifespan=lifespan)

class GenerateRequest(BaseModel):
    prompt: str = ""
    max_tokens: int = 4096
    temperature: float = 0.6
    repetition_penalty: float = 1.15
    schema_dict: dict | None = None

@app.post("/generate")
async def generate(req: GenerateRequest):
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "pending", "response": None, "error": None}
    
    async def run_inference():
        try:
            # First prepare the structured outputs if a schema exists
            struct_out = None
            if req.schema_dict and HAS_STRUCTURED_PARAMS:
                struct_out = StructuredOutputsParams(json=req.schema_dict)

            # Create the correct sampling parameters with the new API
            sampling_params = SamplingParams(
                max_tokens=req.max_tokens,
                temperature=req.temperature,
                repetition_penalty=req.repetition_penalty,
                structured_outputs=struct_out
            )

            request_id = str(uuid.uuid4())
            
            print(f"\n🧠 [NUEVO JOB] Recibido request: {job_id[:8]}...")
            
            final_output = None
            async for output in engine.generate(req.prompt, sampling_params, request_id):
                final_output = output
                
            text = final_output.outputs[0].text
            jobs[job_id] = {"status": "done", "response": text, "error": None}

            # Model reasoning stream output
            print(f"\n✅ [JOB COMPLETADO {job_id[:8]}]")
            print("👇 --- OUTPUT DEL LLM (Reasoning + JSON) --- 👇")
            print(text)
            print("👆 ----------------------------------------- 👆\n")
            
        except Exception as e:
            jobs[job_id] = {"status": "error", "response": None, "error": str(e)}
            print(f"\n❌ [ERROR EN JOB {job_id[:8]}] {str(e)}")
            
    asyncio.create_task(run_inference())
    return {"job_id": job_id}

@app.get("/status/{job_id}")
async def get_status(job_id: str):
    return jobs.get(job_id, {"error": "job_id not found"})

@app.get("/health")
async def health():
    return {"status": "ok"}

print("⚙️ Configuración de FastAPI y vLLM lista. Procede a la Celda 2.")

In [ ]:
import threading
import time
import requests
import uvicorn
from pyngrok import ngrok

def iniciar_servidor_interno():
    # Uvicorn will boot using the 'app' variable created in Cell 1
    # Keep log_level="info" to see incoming HTTP requests
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Preventive port cleanup
print("🧹 Asegurando que el puerto 8000 esté libre...")
os.system("fuser -k 8000/tcp >/dev/null 2>&1")

ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")
public_url = ngrok.connect(8000)
print(f"\n🌐 NGROK ABIERTO EN: {public_url.public_url}")

# Launch the server in background
server_thread = threading.Thread(target=iniciar_servidor_interno, daemon=True)
server_thread.start()

print("⏳ Esperando a que FastAPI acople el motor de vLLM...")

server_ready = False
for i in range(150):
    try:
        response = requests.get("http://localhost:8000/health", timeout=2)
        if response.status_code == 200:
            print(f"\n🟢 ¡TODO LISTO! La API está corriendo de forma limpia.")
            server_ready = True
            break
    except requests.exceptions.ConnectionError:
        pass 
    time.sleep(2)


if not server_ready:
    print("\n❌ Se alcanzó el tiempo límite. Revisa los logs de Uvicorn arriba.")
else:
    print("\n🟢 LA CELDA TERMINÓ, PERO LA API SIGUE CORRIENDO EN EL FONDO.")


In [ ]:
!git clone https://github.com/thunlp/TritonBench.git tritonbench_repo

In [ ]:
!unzip ptx_only_folder.zip

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
!unzip tritonbench_repo.zip

In [ ]:
import json
import requests
import re
import time

def extract_func_name(instruction):
    # Extract the function name based on the instruction signature
    wrapper_info = instruction.split("Wrapper Entry Information:")[-1].strip()
    func_name = wrapper_info.split("(")[0].replace("def ", "").strip()
    return func_name

def generate_raw_baseline():
    print("📥 Loading TritonBench dataset...")
    with open("tritonbench_repo/data/TritonBench_T_simp_alpac_v1.json", "r") as f:
        dataset = json.load(f)
        
    predictions = []
    
    print(f"🚀 Starting Zero-Shot evaluation of {len(dataset)} kernels...")
    for i, item in enumerate(dataset):
        instruction = item["instruction"]
        func_name = extract_func_name(instruction)
        
        # RAW ZERO-SHOT PROMPT
        prompt = f"""You are an expert GPU programmer. Write a highly optimized PyTorch and Triton kernel for the following instruction.\nInstruction: {instruction}\nRespond ONLY with valid Python code containing the @triton.jit kernel and the wrapper function. Do not include markdown formatting like ```python, just the raw code."""

        payload = {
            "prompt": prompt,
            "max_tokens": 2048,
            "temperature": 0.1 # Low temperature for more deterministic code
        }
        
        try:
            # Send request to local vLLM server
            response = requests.post("http://localhost:8000/generate", json=payload).json()
            job_id = response["job_id"]
            
            # Wait for response
            while True:
                status = requests.get(f"http://localhost:8000/status/{job_id}").json()
                if status["status"] == "done":
                    gen_code = status["response"]
                    break
                elif status["status"] == "error":
                    gen_code = "# Internal generation error"
                    break
                time.sleep(0.5)
                
            # Basic cleanup in case the model insists on using markdown
            gen_code = gen_code.replace("```python", "").replace("```", "").strip()
            
            # Exact format expected by 0_call_acc.py
            pred_obj = {
                "instruction": instruction,
                "predict": gen_code
            }
            predictions.append(pred_obj)
            print(f"[{i+1}/{len(dataset)}] ✅ Python code generated for: {func_name}")
            
        except Exception as e:
            print(f"[{i+1}/{len(dataset)}] ❌ Failed to connect/generate {func_name}: {e}")
            
    # Save results
    print("\n💾 Saving predictions_baseline.jsonl...")
    with open("predictions_baseline.jsonl", "w") as f:
        for p in predictions:
            f.write(json.dumps(p) + "\n")
    print("Generation completed!")

generate_raw_baseline()

In [ ]:
!cd tritonbench_repo && python EVAL/eval_T/0_call_acc.py --source ../predictions_baseline.jsonl --target temp_baseline --GPUs 0
!cd tritonbench_repo && python EVAL/eval_T/1_exe_acc.py --folder temp_baseline --GPUs 0

In [ ]:
import os
import signal
import psutil

# Find and kill any process using port 8000
for proc in psutil.process_iter(["pid", "name"]):
    try:
        connections = proc.connections(kind="inet")
        for conn in connections:
            if conn.laddr.port == 8000:
                print(f"Killing process {proc.info['name']} (PID: {proc.info['pid']}) using port 8000")
                os.kill(proc.info["pid"], signal.SIGKILL)
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        continue
